In [0]:
# NBA SHOT IQ - BRONZE
# Preparacion de la arquitectura Medallion

catalogo = spark.sql("SELECT current_catalog()").first()[0]

spark.sql("CREATE SCHEMA IF NOT EXISTS nba_shot_iq")

print("Catalogo activo:", catalogo)
print("Esquema creado: nba_shot_iq")
print("Listo para cargar los datos originales")

Catalogo activo: workspace
Esquema creado: nba_shot_iq
Listo para cargar los datos originales


In [0]:
import requests

url = "https://api.github.com/repos/sportsdataverse/sportsdataverse-data/releases/tags/nba_stats_shots"

respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()

archivos = respuesta.json()["assets"]

for anio in [2024, 2025, 2026]:
    nombre = f"shots_{anio}.parquet"
    encontrados = [
        archivo for archivo in archivos
        if archivo["name"] == nombre
    ]

    if encontrados:
        archivo = encontrados[0]
        print(anio, nombre, archivo["size"], "bytes")
        print(archivo["browser_download_url"])
    else:
        print("NO ENCONTRADO:", nombre)

2024 shots_2024.parquet 4271538 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2024.parquet
2025 shots_2025.parquet 4297616 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2025.parquet
2026 shots_2026.parquet 4261369 bytes
https://github.com/sportsdataverse/sportsdataverse-data/releases/download/nba_stats_shots/shots_2026.parquet


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.nba_shot_iq")

spark.sql("SHOW TABLES IN workspace.nba_shot_iq").show()

+--------+---------+-----------+
|database|tableName|isTemporary|
+--------+---------+-----------+
+--------+---------+-----------+



In [0]:
import requests

url = "https://api.github.com/repos/sportsdataverse/nba-data/releases"

respuesta = requests.get(url, timeout=30)
respuesta.raise_for_status()

releases = respuesta.json()

for release in releases:
    for archivo in release.get("assets", []):
        nombre = archivo["name"]
        if nombre in [
            "shots_2024.parquet",
            "shots_2025.parquet",
            "shots_2026.parquet"
        ]:
            print(nombre, archivo["browser_download_url"])

---------------------------------------------------------------------------
HTTPError                                 Traceback (most recent call last)
File <command-8657067588947001>, line 6
      3 url = "https://api.github.com/repos/sportsdataverse/nba-data/releases"
      5 respuesta = requests.get(url, timeout=30)
----> 6 respuesta.raise_for_status()
      8 releases = respuesta.json()
     10 for release in releases:

File /databricks/python/lib/python3.12/site-packages/requests/models.py:1026, in Response.raise_for_status(self)
   1021     http_error_msg = (
   1022         f"{self.status_code} Server Error: {reason} for url: {self.url}"
   1023     )
   1025 if http_error_msg:
-> 1026     raise HTTPError(http_error_msg, response=self)

HTTPError: 404 Client Error: Not Found for url: https://api.github.com/repos/sportsdataverse/nba-data/releases

In [0]:
import requests

url = "https://api.github.com/repos/sportsdataverse/nba_data/releases"

respuesta = requests.get(url, timeout=30)

print("Estado HTTP:", respuesta.status_code)

if respuesta.ok:
    releases = respuesta.json()
    print("Releases encontrados:", len(releases))

    for release in releases:
        for archivo in release.get("assets", []):
            nombre = archivo["name"]
            if nombre in [
                "shots_2024.parquet",
                "shots_2025.parquet",
                "shots_2026.parquet"
            ]:
                print(nombre, archivo["browser_download_url"])
else:
    print(respuesta.text[:500])

Estado HTTP: 404
{"message":"Not Found","documentation_url":"https://docs.github.com/rest/releases/releases#list-releases","status":"404"}


In [0]:
import requests

base = (
    "https://github.com/sportsdataverse/"
    "sportsdataverse-data/releases/download/nba_stats_shots/"
)

for anio in [2024, 2025, 2026]:
    url = base + f"shots_{anio}.parquet"
    respuesta = requests.get(
        url,
        headers={"Range": "bytes=0-3"},
        stream=True,
        timeout=30
    )
    print(f"shots_{anio}.parquet: HTTP {respuesta.status_code}")
    respuesta.close()

shots_2024.parquet: HTTP 206
shots_2025.parquet: HTTP 206
shots_2026.parquet: HTTP 206


In [0]:
import requests
import os

base = (
    "https://github.com/sportsdataverse/"
    "sportsdataverse-data/releases/download/nba_stats_shots/"
)

carpeta = "/Volumes/workspace/nba_shot_iq/raw_nba_shots"

for anio in [2024, 2025, 2026]:
    nombre = f"shots_{anio}.parquet"
    destino = f"{carpeta}/{nombre}"

    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        print(f"Ya existe: {nombre}")
        continue

    print(f"Descargando {nombre}...")

    with requests.get(base + nombre, stream=True, timeout=120) as r:
        r.raise_for_status()

        with open(destino, "wb") as archivo:
            for bloque in r.iter_content(chunk_size=1024 * 1024):
                if bloque:
                    archivo.write(bloque)

    print(f"Guardado: {nombre} ({os.path.getsize(destino):,} bytes)")

print("Descargas terminadas.")

Descargando shots_2024.parquet...
Guardado: shots_2024.parquet (4,271,538 bytes)
Descargando shots_2025.parquet...
Guardado: shots_2025.parquet (4,297,616 bytes)
Descargando shots_2026.parquet...
Guardado: shots_2026.parquet (4,261,369 bytes)
Descargas terminadas.


In [0]:
from pyspark.sql import functions as F

ruta = "/Volumes/workspace/nba_shot_iq/raw_nba_shots"

df_bronze = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(
        f"{ruta}/shots_2024.parquet",
        f"{ruta}/shots_2025.parquet",
        f"{ruta}/shots_2026.parquet"
    )
)

print("Total de registros:", df_bronze.count())
print("Total de columnas:", len(df_bronze.columns))

df_bronze.printSchema()

display(df_bronze.limit(10))

Total de registros: 703805
Total de columnas: 19
root
 |-- game_id: string (nullable = true)
 |-- season: integer (nullable = true)
 |-- season_type_id: string (nullable = true)
 |-- period: long (nullable = true)
 |-- clock: string (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team_tricode: string (nullable = true)
 |-- person_id: long (nullable = true)
 |-- player_name: string (nullable = true)
 |-- action_type: string (nullable = true)
 |-- sub_type: string (nullable = true)
 |-- shot_result: string (nullable = true)
 |-- shot_value: long (nullable = true)
 |-- shot_distance: long (nullable = true)
 |-- x_legacy: long (nullable = true)
 |-- y_legacy: long (nullable = true)
 |-- description: string (nullable = true)
 |-- score_home: string (nullable = true)
 |-- score_away: string (nullable = true)



game_id,season,season_type_id,period,clock,team_id,team_tricode,person_id,player_name,action_type,sub_type,shot_result,shot_value,shot_distance,x_legacy,y_legacy,description,score_home,score_away
0022400001,2025,2,1,PT11M43.00S,1610612737,ATL,1642258,Risacher,Missed Shot,Jump Shot,Missed,3,27,-168,205,MISS Risacher 27' 3PT Jump Shot,,
0022400001,2025,2,1,PT11M38.00S,1610612737,ATL,1630552,Johnson,Missed Shot,Driving Floating Bank Jump Shot,Missed,2,14,-136,-1,MISS Johnson 14' Driving Floating Bank Jump Shot,,
0022400001,2025,2,1,PT11M24.00S,1610612738,BOS,1628401,White,Missed Shot,Jump Shot,Missed,3,28,-181,218,MISS White 28' 3PT Jump Shot,,
0022400001,2025,2,1,PT10M50.00S,1610612737,ATL,1630552,Johnson,Made Shot,Jump Shot,Made,3,26,157,203,Johnson 26' 3PT Jump Shot (3 PTS) (Wallace 1 AST),0,3
0022400001,2025,2,1,PT10M35.00S,1610612738,BOS,1627759,Brown,Made Shot,Pullup Jump shot,Made,3,27,102,253,Brown 27' 3PT Pullup Jump Shot (3 PTS) (White 1 AST),3,3
0022400001,2025,2,1,PT10M06.00S,1610612738,BOS,1627759,Brown,Missed Shot,Jump Shot,Missed,3,24,-228,85,MISS Brown 24' 3PT Jump Shot,,
0022400001,2025,2,1,PT09M47.00S,1610612737,ATL,1630811,Wallace,Missed Shot,Jump Shot,Missed,3,25,-176,184,MISS Wallace 25' 3PT Jump Shot,,
0022400001,2025,2,1,PT09M44.00S,1610612737,ATL,203991,Capela,Made Shot,Putback Layup Shot,Made,2,3,-25,8,Capela 3' Putback Layup (2 PTS),3,5
0022400001,2025,2,1,PT09M35.00S,1610612738,BOS,1627759,Brown,Made Shot,Driving Layup Shot,Made,2,1,5,13,Brown 1' Driving Layup (5 PTS) (White 2 AST),5,5
0022400001,2025,2,1,PT09M23.00S,1610612737,ATL,1642258,Risacher,Made Shot,Cutting Layup Shot,Made,2,2,10,13,Risacher 2' Cutting Layup Shot (2 PTS) (Wallace 2 AST),5,7


In [0]:
# Guardar los datos originales en la capa Bronze
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.nba_shot_iq.bronze_shots")
)

# Comprobar que la tabla se creó correctamente
df_verificacion = spark.table(
    "workspace.nba_shot_iq.bronze_shots"
)

print("BRONZE CREADA CORRECTAMENTE")
print("Registros:", df_verificacion.count())
print("Columnas:", len(df_verificacion.columns))

display(
    df_verificacion.groupBy("season")
    .count()
    .orderBy("season")
)

BRONZE CREADA CORRECTAMENTE
Registros: 703805
Columnas: 19


season,count
2024,233789
2025,235140
2026,234876


In [0]:
from pyspark.sql import functions as F

# Leer la tabla Bronze original
bronze = spark.table("workspace.nba_shot_iq.bronze_shots")

# Eliminar duplicados exactos
silver = bronze.dropDuplicates(bronze.columns)

# Normalizar textos de categorías
for columna in [
    "team_tricode",
    "player_name",
    "action_type",
    "sub_type",
    "shot_result"
]:
    silver = silver.withColumn(
        columna,
        F.trim(F.col(columna))
    )

# Convertir marcadores vacíos a nulos
for columna in ["score_home", "score_away"]:
    silver = silver.withColumn(
        columna,
        F.when(
            F.trim(F.col(columna)) == "",
            F.lit(None)
        ).otherwise(F.trim(F.col(columna)))
    )

# Crear variables para el análisis
silver = (
    silver
    .withColumn(
        "is_made",
        F.when(F.lower(F.col("shot_result")) == "made", 1)
         .when(F.lower(F.col("shot_result")) == "missed", 0)
    )
    .withColumn(
        "shot_points",
        F.when(F.col("is_made") == 1, F.col("shot_value"))
         .otherwise(F.lit(0))
    )
    .withColumn(
        "shot_zone",
        F.when(F.col("shot_distance") < 5, "Cerca del aro")
         .when(F.col("shot_distance") < 15, "Distancia media")
         .otherwise("Distancia larga")
    )
)

# Guardar la tabla Silver en Delta
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.nba_shot_iq.silver_shots")
)

# Verificar resultados
total_bronze = bronze.count()
total_silver = spark.table(
    "workspace.nba_shot_iq.silver_shots"
).count()

print("BRONZE:", total_bronze)
print("SILVER:", total_silver)
print("DUPLICADOS ELIMINADOS:", total_bronze - total_silver)

BRONZE: 703805
SILVER: 703781
DUPLICADOS ELIMINADOS: 24


In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.nba_shot_iq.silver_shots")

# 1. Eficiencia por temporada
gold_temporadas = (
    silver.groupBy("season")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct"),
        F.sum("shot_points").alias("puntos_generados"),
        F.round(F.avg("shot_distance"), 2).alias("distancia_promedio")
    )
)

# 2. Eficiencia por zona
gold_zonas = (
    silver.groupBy("season", "shot_zone")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct")
    )
)

# 3. Eficiencia por jugador y temporada
gold_jugadores = (
    silver.groupBy("season", "person_id", "player_name")
    .agg(
        F.count("*").alias("intentos"),
        F.sum("is_made").alias("encestados"),
        F.round(F.avg("is_made") * 100, 2).alias("efectividad_pct"),
        F.sum("shot_points").alias("puntos_generados")
    )
)

# Guardar las tres tablas Gold en Delta
for nombre, df in [
    ("gold_temporadas", gold_temporadas),
    ("gold_zonas", gold_zonas),
    ("gold_jugadores", gold_jugadores)
]:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.nba_shot_iq.{nombre}")
    )
    print("Tabla creada:", nombre)

# Mostrar resultados
display(
    spark.table("workspace.nba_shot_iq.gold_temporadas")
    .orderBy("season")
)

Tabla creada: gold_temporadas
Tabla creada: gold_zonas
Tabla creada: gold_jugadores


season,intentos,encestados,efectividad_pct,puntos_generados,distancia_promedio
2024,233776,110696,47.35,255049,14.04
2025,235132,109665,46.64,254895,14.51
2026,234873,110338,46.98,255614,14.3


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.nba_shot_iq.bronze_shots")
silver = spark.table("workspace.nba_shot_iq.silver_shots")
gold = spark.table("workspace.nba_shot_iq.gold_temporadas")

resultados = []

def verificar(nombre, condicion):
    estado = "PASS" if condicion else "FAIL"
    resultados.append((nombre, estado))
    print(f"{estado}: {nombre}")

# Volumen y estructura
verificar("Bronze contiene 703805 registros", bronze.count() == 703805)
verificar("Bronze conserva 19 columnas", len(bronze.columns) == 19)
verificar("Silver contiene 703781 registros", silver.count() == 703781)
verificar("Se eliminaron 24 duplicados", bronze.count() - silver.count() == 24)

# Calidad de los datos
verificar(
    "Sin duplicados exactos en Silver",
    silver.count() == silver.dropDuplicates(bronze.columns).count()
)

verificar(
    "Sin distancias negativas",
    silver.filter(F.col("shot_distance") < 0).count() == 0
)

verificar(
    "Indicador is_made valido",
    silver.filter(
        F.col("is_made").isNull() |
        (~F.col("is_made").isin(0, 1))
    ).count() == 0
)

verificar(
    "Temporadas correctas",
    silver.filter(~F.col("season").isin(2024, 2025, 2026)).count() == 0
)

# Consistencia entre Silver y Gold
verificar(
    "Gold conserva todos los intentos",
    gold.agg(F.sum("intentos")).first()[0] == silver.count()
)

verificar(
    "Gold contiene tres temporadas",
    gold.count() == 3
)

# Resumen
aprobadas = sum(1 for _, estado in resultados if estado == "PASS")
print(f"\nVALIDACIONES APROBADAS: {aprobadas}/{len(resultados)}")

PASS: Bronze contiene 703805 registros
PASS: Bronze conserva 19 columnas
PASS: Silver contiene 703781 registros
PASS: Se eliminaron 24 duplicados
PASS: Sin duplicados exactos en Silver
PASS: Sin distancias negativas
PASS: Indicador is_made valido
PASS: Temporadas correctas
PASS: Gold conserva todos los intentos
PASS: Gold contiene tres temporadas

VALIDACIONES APROBADAS: 10/10
